In [ ]:


import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# Convert MNIST Image Files into a Tensor of 4-Dimensions (# of images, Height, Width,colour channels)
transform = transforms.ToTensor()

In [ ]:
#train Data
train_data = datasets.MNIST(root='/cnn_data',train=True,download=True, transform=transform)

In [ ]:
#Test Data
test_data = datasets.MNIST(root='/cnn_data',train=False,download=True, transform=transform)

In [ ]:
train_data

In [ ]:
test_data

In [ ]:
#Create a small batch of images - 10
train_loader = DataLoader(train_data, batch_size=10, shuffle=True)
test_loader = DataLoader(test_data, batch_size=10, shuffle=False)
test_load_everything = DataLoader(test_data, batch_size=10000, shuffle=False)


In [ ]:
conv1 = nn.Conv2d(1,6,3,1)
conv2 = nn.Conv2d(6,16,3,1)

In [ ]:
for i, (x_train, y_train) in enumerate(train_data):
    break

In [ ]:
x_train.shape

In [ ]:
x = x_train.view(1,1,28,28)

In [ ]:
x = F.relu(conv1(x))

In [ ]:
x.shape

In [ ]:
x = F.max_pool2d(x, 2,2)

In [ ]:
x.shape

In [ ]:
x = F.relu(conv2(x))

In [ ]:
x.shape

In [ ]:
x = F.max_pool2d(x, 2,2)

In [ ]:
x.shape

In [ ]:
#Model
class convolutionalnetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(1,6,3,1)
    self.conv2 = nn.Conv2d(6,16,3,1)
    #fully connected layer
    self.fc1 = nn.Linear(5*5*16, 120)
    self.fc2 = nn.Linear(120, 84)
    self.fc3 = nn.Linear(84, 10)

  def forward(self, X):
    X = F.relu(self.conv1(X))
    X = F.max_pool2d(X, 2,2) #2*2 Kernal and stride of 2
    X = F.relu(self.conv2(X))
    X = F.max_pool2d(X, 2,2)

    X =X.view(-1, 16*5*5)

    #Fully connected Layer

    X = F.relu(self.fc1(X))
    X = F.relu(self.fc2(X))
    X = self.fc3(X)

    return F.log_softmax(X, dim=1)


In [ ]:
#Create an instance of our Model

torch.manual_seed(41)
model = convolutionalnetwork()
model

In [ ]:
#Los Function Optimizer
criterion =nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001) #smaller learning the learning and longer the train


In [ ]:
import time
import mlflow
import mlflow.pytorch

mlflow.set_experiment("mnist-cnn")

# Create Variables To Tracks Things
epochs = 5
train_losses = []
test_losses = []
train_correct = []
test_correct = []

with mlflow.start_run(run_name="cnn-lenet-style"):   # MLFLOW
  mlflow.log_params({"epochs": epochs, "batch_size": 10, "learning_rate": 0.001, "optimizer": "Adam"})   # MLFLOW
  start_time = time.time()

  # For Loop of Epochs
  for i in range(epochs):
    trn_corr = 0
    tst_corr = 0
    running_loss = 0.0   # MLFLOW
    test_loss_total = 0.0   # MLFLOW

    # Train
    for b,(X_train, y_train) in enumerate(train_loader):
      b+=1 # start our batches at 1
      y_pred = model(X_train) # get predicted values from the training set. Not flattened 2D
      loss = criterion(y_pred, y_train) # how off are we? Compare the predictions to correct answers in y_train

      predicted = torch.max(y_pred.data, 1)[1] # add up the number of correct predictions. Indexed off the first point
      batch_corr = (predicted == y_train).sum() # how many we got correct from this batch. True = 1, False=0, sum those up
      trn_corr += batch_corr # keep track as we go along in training.
      running_loss += loss.item()   # MLFLOW

      # Update our parameters
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      # Print out some results
      if b%600 == 0:
        print(f'Epoch: {i}  Batch: {b}  Loss: {loss.item()}')

    train_losses.append(loss)
    train_correct.append(trn_corr)

    # Test
    with torch.no_grad(): #No gradient so we don't update our weights and biases with test data
      for b,(X_test, y_test) in enumerate(test_loader):
        y_val = model(X_test)
        predicted = torch.max(y_val.data, 1)[1] # Adding up correct predictions
        tst_corr += (predicted == y_test).sum() # T=1 F=0 and sum away
        test_loss_total += criterion(y_val, y_test).item()   # MLFLOW

    loss = criterion(y_val, y_test)
    test_losses.append(loss)
    test_correct.append(tst_corr)

    # MLFLOW: log this epoch's metrics
    mlflow.log_metrics({
        "train_loss": running_loss / len(train_loader),
        "train_accuracy": trn_corr.item() / len(train_data) * 100,
        "test_loss": test_loss_total / len(test_loader),
        "test_accuracy": tst_corr.item() / len(test_data) * 100,
    }, step=i)

  total = time.time() - start_time
  print(f'Training Took: {total/60} minutes!')
  mlflow.log_metric("training_time_minutes", total/60)   # MLFLOW

  # MLFLOW: final accuracy on the full test set, then save the model
  with torch.no_grad():
    correct = 0
    for X_test, y_test in test_load_everything:
      y_val = model(X_test)
      predicted = torch.max(y_val, 1)[1]
      correct += (predicted == y_test).sum()
  mlflow.log_metric("final_test_accuracy", correct.item()/len(test_data)*100)
  mlflow.pytorch.log_model(model, "model")

In [ ]:
#Graph the loss at each epoch

# Graph the loss at epoch
train_losses = [tl.item() for tl in train_losses]
plt.plot(train_losses, label="Training Loss")
plt.plot(test_losses, label="Validation Loss")
plt.title("Loss at Epoch")
plt.legend()

In [ ]:

# graph the accuracy at the end of each epoch
plt.plot([t/600 for t in train_correct], label="Training Accuracy")
plt.plot([t/100 for t in test_correct], label="Validation Accuracy")
plt.title("Accuracy at the end of each Epoch")
plt.legend()


In [ ]:

with torch.no_grad():
  correct = 0
  for X_test, y_test in test_load_everything:
    y_val = model(X_test)
    predicted = torch.max(y_val, 1)[1]
    correct += (predicted == y_test).sum()


In [ ]:
# Did for correct
correct.item()/len(test_data)*100

In [ ]:

# Grab an image
test_data[1978] # Tensor with an image in it...at end, it shows the label

In [ ]:
# Grab just the data
test_data[1978][0]

In [ ]:

# Reshape it
test_data[1978][0].reshape(28,28)

In [ ]:
# Show the image
plt.imshow(test_data[1978][0].reshape(28,28))

In [ ]:
# Pass the image thru our model
model.eval()
with torch.no_grad():
  new_prediction = model(test_data[1978][0].view(1,1,28,28)) # batch size of 1, 1 color channel, 28x28 image

In [ ]:
# Check the new prediction...get probabilities
new_prediction


In [ ]:

new_prediction.argmax()

In [ ]:
torch.save(model, "full_model.pth")

In [ ]:
torch.save(model, "my_model.pt")